# Visual anomaly detection
Ce TP se base sur le papier de Vitjan Zavrtanik , Matej Kristan et Danijel Skcaj : *Reconstruction by inpainting for visual anomaly detection*.

Malheureusement l'article est protégé par le droit d'auteur et scihub n'existe pas. Il n'est donc pas possible de le télécharger.


In [ ]:
! wget https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282/download/420938113-1629952094/mvtec_anomaly_detection.tar.xz
! rm -rf sample_data/
!tar -xf mvtec_anomaly_detection.tar.xz
!pip install -q lightning torchmetrics
!git clone https://github.com/NyxAether/RIAD_models.git

In [ ]:
! nvidia-smi
! find */train/ -name *.png | wc -l

In [ ]:
from pathlib import Path
from tqdm.notebook import tqdm, trange

import cv2
import matplotlib.pyplot as plt
import numpy as np

import lightning
import torch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchmetrics.functional.image import (
  multiscale_structural_similarity_index_measure as ms_ssim,
)

## Fonctions utiles

In [ ]:
RESHAPE_DIM = (256, 256)
device = "cuda" if torch.cuda.is_available() else "cpu"


def load_data(files):
  nb_files = len(files) - len(files) % 10
  data = np.zeros((nb_files, *RESHAPE_DIM, 3), dtype="float32")
  for i in trange(nb_files):
    im = cv2.imread(str(files[i]))
    im = cv2.resize(im, RESHAPE_DIM)
    im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    data[i, :, :, :] = im
  data = data / 255
  # PyTorch attend les canaux juste après la dimension de batch
  data_torch = torch.from_numpy(data).permute(0, 3, 1, 2).contiguous()
  return data_torch


def to_displayable(image: torch.Tensor) -> np.ndarray:
  """Turn a (channels, height, width) tensor into an array imshow can plot."""
  image = image.detach().cpu()
  return image.permute(1, 2, 0).numpy() if image.dim() == 3 else image.numpy()


@torch.no_grad()
def predict(model: nn.Module, data: torch.Tensor, batch_size: int = 5) -> torch.Tensor:
  """Apply a model to a batch of images, batch by batch."""
  model.eval()
  return torch.cat([model(batch.to(device)).cpu() for batch in data.split(batch_size)])


def display(data, is_mask=True):
  shape = data.shape
  if is_mask:
    h, w = shape[0], shape[1]
    fig, ax = plt.subplots(h, w, dpi=150)
    for l in range(h):
      for c in range(w):
        ax[l, c].set_xticks([])
        ax[l, c].set_yticks([])
        ax[l, c].imshow(to_displayable(data[l, c]), cmap="gray")
  else:
    h, w = shape[0], 1
    fig, ax = plt.subplots(h, w, dpi=150)
    for i, axi in enumerate(ax):
      axi.set_xticks([])
      axi.set_yticks([])
      axi.imshow(to_displayable(data[i]), cmap="gray")

## Listes des objets


In [ ]:
dossier_courant = Path(".")
liste_objets = [
  d.name for d in dossier_courant.iterdir() if d.name[0] != "." and not d.is_file()
]
liste_objets.remove("RIAD_models")
for obj in liste_objets:
  print(obj)

## Récupération des données pour un objet
 * Les données d'entrainements
 * Les données de tests incluant des cas anormaux et des cas valides
 * Les masques de détection des anomalies

Toutes les données seront converties en images 256*256.

In [ ]:
objet = "transistor"


def load_all(objet):
  # Données entrainement
  files_train = sorted(list(Path(f"{objet}/train/good/").glob("*.png")))
  nb_files = len(files_train) - len(files_train) % 10
  data = load_data(files_train)

  # Données défectueuses
  defect_path = set(Path(f"{objet}/test/").glob("*")) - set(
    Path(f"{objet}/test/").glob("good")
  )
  files_defect = set()
  for p in defect_path:
    files_defect = files_defect | set(p.glob("*.png"))
  files_defect = sorted(files_defect)
  defect_data = load_data(files_defect)

  # Données valides
  files_good = sorted(list(Path(f"{objet}/test/good/").glob("*.png")))
  goods = load_data(files_good)

  # Masques de vérification pour les données défectueuses
  files_ground = sorted(list(Path(f"{objet}/ground_truth/").glob("*/*.png")))
  ground = load_data(files_ground)
  return data, defect_data, goods, ground


data, defect_data, goods, ground = load_all(objet)

## Masquage des images
La technique de détection d'anomalie repose sur génération de fragments d'image masquée.

Pour ce faire, on génère $n$ masques sur un image donnée en respectant deux règles :
 * L'image est masquée aléatoirement par des blocs de $k\times k$ pixels (en général une puissance de deux, mais en théorie un diviseur de la taille de l'image)
 * Les masques n'ont aucune intersection de masquage et cache l'intégralité de l'image s'ils sont superposés (1 pixel est masqué par au plus un masque, mais au moins un)

In [ ]:
def generate_masks(images_shape: tuple, k: int = 2, n: int = 3) -> torch.Tensor:
  """
  images_shape: shape of a batch of images
  k: size of hidding block
  n: number of masks
  """
  shape = (images_shape[0], *(ax // k for ax in images_shape[-2:]))
  indices = torch.randint(0, n, shape)
  masks = 1 - nn.functional.one_hot(indices, n).float()
  masks = masks.permute(0, 3, 1, 2)
  return nn.functional.interpolate(masks, images_shape[-2:], mode="nearest")


def apply_masks(img: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
  # (b, 1, c, h, w) * (b, m, 1, h, w) -> (b, m, c, h, w)
  return img.unsqueeze(-4) * masks.unsqueeze(-3)


def reconstruct_image(imgs: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
  invert_masks = (1 - masks).unsqueeze(-3)
  return torch.sum(imgs * invert_masks, dim=-4)

`generate_masks` produit des masques aléatoires pour chaque image et `apply_masks` permet de dupliquer une image et appliquer les masques créer sur chacune des copies.
Afin de voir le résultat, appliquons ces méthodes sur deux images.

**Vous pouvez changer la taille des masques $k$ et le nombres de masques $n$**

In [ ]:
# Génération de masques
masks = generate_masks(data[:2].shape, k=16, n=3)
# Application des masques sur les images
data_masked = apply_masks(data[:2], masks)

display(data[:2], is_mask=False)

In [ ]:
display(masks)

In [ ]:
display(data_masked)

Les masques sont exclusifs, c'est à dire qu'il ne recouvre jamais une même partie de l'image. L'objectif du modèle sera de compléter les parties manquantes des données masquées puis de les réassembler.

In [ ]:
inverse_masked = apply_masks(data[:2], 1 - masks)
display(inverse_masked)

## Fonction de coût


La fonction de coût pour la méthode est calculé à partir de la *Gradient magnitude similarity* et de la *Structural similarity index*.

### Gradient magnitude similarity (GMS)


La GMS repose sur le calcul d'un gradient $g(I)$ sur les images considérées :

$$g(I)=\sqrt{(I*h_x)^2+(I*h_y)^2}$$

avec $*$ l'opérateur de convolution, $I$ l'image et le couple $h_x$ et $h_y$ les filtres de Prewitt sur les dimensions $x$ et $y$ correspondant en général à :

$h_x=\begin{bmatrix}
-1 & 0 & +1 \\
-1 & 0 & +1 \\
-1 & 0 & +1
\end{bmatrix}$ et $h_y=\begin{bmatrix}
-1 & -1 & -1 \\
0 & 0 & 0 \\
+1 & +1 & +1
\end{bmatrix}$

$g(I)$ est donc la distance euclidienne entre un gradient horizontal et vertical.


In [ ]:
# Vectorized
def prewitt_filter(img: torch.Tensor, c=1e-9) -> torch.Tensor:
  # Define kernel for x differences
  if img.dim() < 4:
    im = img.reshape((1,) + img.shape)
  else:
    im = img
  kx = (
    torch.tensor([[-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0]])
    .to(im)
    .repeat(3, 1, 1, 1)
  )
  # Define kernel for y differences
  ky = (
    torch.tensor([[-1.0, -1.0, -1.0], [0.0, 0.0, 0.0], [1.0, 1.0, 1.0]])
    .to(im)
    .repeat(3, 1, 1, 1)
  )
  # Perform x convolution
  img_prewittx = nn.functional.conv2d(im, kx, padding="same", groups=3) / 3
  # Perform y convolution
  img_prewitty = nn.functional.conv2d(im, ky, padding="same", groups=3) / 3

  gmm = torch.sqrt(img_prewittx * img_prewittx + img_prewitty * img_prewitty + c)

  if img.dim() < 4:
    gmm = gmm.reshape(img.shape)
  return gmm

In [ ]:
filtered_im = prewitt_filter(data[0])
display(torch.stack([data[0], filtered_im]), is_mask=False)

En utilisant $g(I)$, nous allons calculer la métrique de similarité GMS défini entre deux images $I$ et $I_r$ respectivement l'image originale et l'image reconstruite par l'auto-encodeur :

$GMS(I,I_r)= \frac{2g(I)g(I_r)+c}{g(I)^2+g(I_r)^2+c}$

avec $c$ une constante permettant la stabilité numérique.

In [ ]:
# Vectorized.
def GMS(I: torch.Tensor, Ir: torch.Tensor, c=1e-9) -> torch.Tensor:
  gI = prewitt_filter(I)
  gIr = prewitt_filter(Ir)
  gms = (2 * gI * gIr + c) / (torch.square(gI) + torch.square(gIr) + c)
  return gms


# Tenseur de 1 si identique
print(torch.mean(GMS(data[:2], data[:2]), dim=(-1, -2, -3)))
# Tenseur différent de 1 si différent
print(torch.mean(GMS(data[:2], data[2:4]), dim=(-1, -2, -3)))

### Fonction de coût $L_G$
La GMS étant un tenseur on calcule la fonction de coût $L_G$ comme étant la somme des inverses de la GMS :

$$L_G(I,I_r)=\sum^H_{i=1}\sum^W_{i=1} 1 - GMS(I,I_r)$$

avec $H$ et $W$ la hauteur et la largeur de l'image.

In [ ]:
# Vectorized. Take one image per argument
def L_G(I: torch.Tensor, Ir: torch.Tensor) -> torch.Tensor:
  if I.dim() < 4:
    return torch.mean(1 - GMS(I, Ir))
  else:
    return torch.mean(1 - GMS(I, Ir), dim=(-1, -2, -3))


# Vaut zero en cas d'identité
print(L_G(data[:2], data[:2]))
# Non nul en cas de différence
# mais <dim*dim2
print(L_G(data[:2], data[2:4]))

Plutôt que d'utiliser GMS, nous allons utiliser sa version multiscales $MSGMS(I,I_r)$. C'est à dire volontairement réduire plusieurs fois la taille des images à comparer et appliquer GMS à chaque fois. Les résultats sont ensuite moyennés. Cela va notamment permettre de "*lisser*" les résultats.

Dans le papier original MSGMS est calculé avec un average pooling de (2), jusqu'au huitième de la dimension initiale.

$$ MSMGS(I,Ir) = \frac{1}{|S|}\sum_{s\in S} GMS(I_s,Ir_{s})$$

avec $S$ l'ensemble des formats des images réduites, $I_s$ et $Ir_s$ les images $I$ et $Ir$ redimensionnées au format $s$

In [ ]:
# Vectorized.
def MSGMS(I: torch.Tensor, Ir: torch.Tensor, is_loss=True) -> torch.Tensor:
  total_loss = GMS(I, Ir)

  for _ in range(3):
    I = nn.functional.avg_pool2d(I, 2, stride=2)
    Ir = nn.functional.avg_pool2d(Ir, 2, stride=2)
    total_loss = total_loss + nn.functional.interpolate(
      GMS(I, Ir), total_loss.shape[-2:], mode="bilinear"
    )

  total_loss = total_loss / 4
  if is_loss:
    return 1 - torch.mean(total_loss, dim=(-1, -2, -3))
  else:
    return total_loss


# Version vectorisée
print(MSGMS(data[:2], data[:2]))
print(MSGMS(data[:2], data[2:4]))

### Indice de similarité structurel (SSIM)
L'indice SSIM est une mesure de similarité dédié aux images définie par [Wang et al.](https://www.cns.nyu.edu/pub/eero/wang03-reprint.pdf).

Il prend en compte 3 critères différentes pour établir la similarité :
 * la luminance
 * le contraste
 * la structure générale

Fort heureusement la méthode est déjà implémentée dans le package [**torchmetrics.functional.image.multiscale_structural_similarity_index_measure**](https://lightning.ai/docs/torchmetrics/stable/image/multi_scale_structural_similarity.html), importée ici sous le nom `ms_ssim`. Nous allons donc utiliser la SSIM comme pour la GMS et calculer une fonction de coût $L_S$ qui correspond à la somme de l'inverse des similarités :

$L_S(I,I_r)=\sum^H_{i=1}\sum^W_{i=1} 1 - SSIM(I,I_r)$

In [ ]:
# Vectorized.
def L_S(I: torch.Tensor, Ir: torch.Tensor, max_val=1.0) -> torch.Tensor:
  # Max val is the maximum difference between values allowed
  matrix_ssim = ms_ssim(I, Ir, data_range=max_val, reduction="none")
  return 1 - matrix_ssim


# Vaut zero en cas d'identité
print(L_S(data[:2], data[:2]))
# Non nul en cas de différence
print(L_S(data[:2], data[2:4]))

### Fonction de coût finale
La fonction de coût $L$ que nous utiliserons sera juste la somme pondérée des fonctions de coût $L_G$ et $L_S$ :

$L=λ_G L_G + λ_S L_S + L2$

avec $λ_G,λ_S$ les pondérations des fonctions de coût respectives $L_G, L_S$ et $L2$ la regularisation de l'autoencodeur.

In [ ]:
def loss_vad(
  y: torch.Tensor, y_hat: torch.Tensor, lambda_G: float = 1, lambda_S: float = 1
) -> torch.Tensor:
  return torch.sum(
    lambda_G * MSGMS(y, y_hat)
    + lambda_S * L_S(y, y_hat)
    + torch.mean((y - y_hat) ** 2, dim=(-1, -2, -3))
  )


# Vaut zero en cas d'identité
print(loss_vad(data[:2], data[:2]))
# Non nul en cas de différence
print(loss_vad(data[:2], data[2:4]))

## Creation du modèle
Le modèle RIAD applique donc la techinique de masquage pour retrouver les anomalies pouvant être présentes dans une image.

Pendant l'entrainement, on applique l'algorithme suivant :
 * Choix aléatoire d'une taille de bloc $k$ pour les masques (en général dans les valeurs 2,4,8,16
 * Génération aléatoire et applications des masks sur les images
 * Pour chaque image masqué, l'auto-encodeur tente de reproduire l'image initiale.
 * Les parties masquées générées par l'auto-encodeur sont réassemblées pour former une seule image qui est ensuite comparée à l'image d'origine

Le reste est standard

In [ ]:
class RIAD(lightning.LightningModule):
  def __init__(self, model: nn.Module, learning_rate: float = 1e-3) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model

  def forward(self, x):
    return self.model(x)

  def training_step(self, batch, batch_index):
    (x,) = batch
    k = torch.randint(1, 4, (1, 1))[0, 0]
    masks = generate_masks(x.shape, 2**k, 3).to(x)
    nb_masks = masks.shape[1]
    masked = apply_masks(x, masks)

    partial_xs = []
    for i in range(nb_masks):
      partial_xs.append(self(masked[:, i]))  # Forward pass

    x_r = reconstruct_image(torch.stack(partial_xs, dim=1), masks)
    loss = loss_vad(x, x_r)
    loss_g = MSGMS(x, x_r).mean()
    loss_s = L_S(x, x_r).mean()
    l2 = torch.mean((x - x_r) ** 2)

    # Log a dict mapping metric names to current value
    self.log_dict(
      {"loss": loss, "lossg": loss_g, "loss_s": loss_s, "l2": l2},
      on_step=False,
      on_epoch=True,
      prog_bar=True,
    )
    return loss

  def configure_optimizers(self):
    return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

  @torch.no_grad()
  def map_error(self, data):
    """
    Permet de générer la heatmap à partir du score MSGMS
    """
    mask_sizes = (2, 4, 8, 16)
    self.eval()
    data = data.to(self.device)
    Ga = torch.zeros_like(data)
    avg_filter = torch.ones(1, 3, 21, 21).to(data) / (21 * 21 * 3)
    for k in mask_sizes:
      masks = generate_masks(data.shape, k, 3).to(data)
      nb_masks = masks.shape[1]
      masked = apply_masks(data, masks)

      partial_xs = []
      for i in range(nb_masks):
        partial_xs.append(self(masked[:, i]))  # Forward pass

      x_r = reconstruct_image(torch.stack(partial_xs, dim=1), masks)
      Ga = Ga + 1 - MSGMS(data, x_r, False)
    Ga = Ga / len(mask_sizes)
    Ga = nn.functional.conv2d(Ga, avg_filter, padding="same")
    return Ga

### Réseau UNet

In [ ]:
def downsample() -> nn.MaxPool2d:
  return nn.MaxPool2d(kernel_size=(2, 2))


def upsample() -> nn.Upsample:
  return nn.Upsample(scale_factor=2)


def semiblock(in_channels, feature_maps):
  conv = nn.Conv2d(in_channels, feature_maps, 3, padding="same")
  batch_norm = nn.BatchNorm2d(feature_maps)
  return nn.Sequential(conv, nn.ReLU(), batch_norm, nn.ReLU(), nn.Dropout(0.1))


def block(in_channels, feature_maps):
  return nn.Sequential(
    semiblock(in_channels, feature_maps), semiblock(feature_maps, feature_maps)
  )


class UNet(nn.Module):
  """U-net: an encoder, a decoder, and skip connections between the two."""

  def __init__(
    self, in_channels: int = 3, starting_depth: int = 64, steps: int = 5
  ) -> None:
    super().__init__()
    depths = [min(512, starting_depth * 2**i) for i in range(steps)]

    self.downsample = downsample()
    self.upsample = upsample()

    self.encoder = nn.ModuleList(
      [
        block(depths[i - 1] if i else in_channels, depth)
        for i, depth in enumerate(depths)
      ]
    )

    # À chaque niveau du décodeur, l'encodage de même résolution est concaténé
    # au tenseur sur-échantillonné, d'où le nombre de canaux en entrée
    self.decoder = nn.ModuleList(
      [
        block(depth + previous_depth, depth)
        for depth, previous_depth in zip(depths[-2::-1], depths[::-1])
      ]
    )

    self.outputs = nn.Conv2d(depths[0], 3, 1)

  def forward(self, inputs: torch.Tensor) -> torch.Tensor:
    encodings = []
    for i, encoder_block in enumerate(self.encoder):
      encodings.append(encoder_block(self.downsample(encodings[-1]) if i else inputs))

    x = encodings[-1]
    for decoder_block, encoding in zip(self.decoder, encodings[-2::-1]):
      x = decoder_block(torch.cat([encoding, self.upsample(x)], dim=1))

    return torch.sigmoid(self.outputs(x))


def unet(pretrained_weights=None, input_size=(3, 256, 256), starting_depth=64, steps=5):
  model = RIAD(UNet(input_size[0], starting_depth, steps))

  # print(ModelSummary(model, max_depth=-1))

  if pretrained_weights:
    model.load_state_dict(torch.load(pretrained_weights))

  return model


# Build model
model = unet()

## Apprentissage
__ATTENTION__ : L'apprentissage peut être très long.

Vous pouvez descendre directement à la partie `Visualisation avec des modèles pré-entrainés` si vous souhaitez juste voir le résultat.

In [ ]:
callbacks = [
  EarlyStopping(
    monitor="loss",
    patience=20,
    check_on_train_epoch_end=True,
  ),
  ModelCheckpoint(monitor="loss", mode="min", save_top_k=1, filename="best_model"),
]
train_loader = DataLoader(TensorDataset(data), batch_size=5, shuffle=True)
trainer = lightning.Trainer(
  max_epochs=1000,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="riad"),
  callbacks=callbacks,
)
trainer.fit(model, train_loader)

# Rechargement des poids du meilleur modèle
model.load_state_dict(
  torch.load(trainer.checkpoint_callback.best_model_path)["state_dict"]
)

## Évaluations
Regardons les résultats de notre apprentissage

In [ ]:
def error_vis(im, pred, err_map, ground=None, threshold=0.5, name=None):
  fig, ax = plt.subplots(
    len(im), 3 if ground is None else 4, figsize=(9, 4 * len(im)), squeeze=False
  )
  for i in range(len(im)):
    for axi in ax[i]:
      axi.set_xticklabels([])
      axi.set_yticklabels([])
    ax[i, 0].imshow(to_displayable(im[i]))
    ax[i, 0].imshow(to_displayable(err_map[i][0, 0]), alpha=0.5, cmap="jet")

    ax[i, 1].imshow(to_displayable(pred[i]))
    ax[i, 2].imshow(to_displayable(im[i]))

    if ground is not None:
      ax[i, 3].imshow(to_displayable(im[i]))
      ax[i, 3].imshow(to_displayable(ground[i]), cmap="jet", alpha=0.3)
  fig.tight_layout()
  if name != None:
    fig.savefig(f"{name}.png")
  else:
    fig.show()


def display_err(objet: str, max_object: int = 5, good_data=False):
  model = unet(f"RIAD_models/best_model_{objet}.pt")
  model.to(device)
  _, defect_data, good, ground = load_all(objet)
  defect_data = defect_data[:max_object]
  ground = ground[:max_object]
  good = good[:max_object]
  errors = []
  if good_data:
    for d in tqdm(good):
      errors.append(model.map_error(d[None, :]))

    res = predict(model, good)
    error_vis(good, res, errors, name=objet)
  else:
    for d in tqdm(defect_data):
      errors.append(model.map_error(d[None, :]))

    res = predict(model, defect_data)
    error_vis(defect_data, res, errors, ground, name=objet)

### Données avec anomalies

In [ ]:
errors = []
for d in tqdm(defect_data):
  errors.append(model.map_error(d[None, :]))

res = predict(model, defect_data)

In [ ]:
error_vis(defect_data, res, errors, ground)

### Données sans anomalies

In [ ]:
err_goods = []
for d in tqdm(goods):
  err_goods.append(model.map_error(d[None, :]))

res_goods = predict(model, goods)

In [ ]:
error_vis(goods, res_goods, err_goods)

## Visualisation avec des modèles pré-entrainés

### Données avec anomalies

In [ ]:
objet = "carpet"
display_err(objet)

### Données sans anomalie


In [ ]:
display_err(objet, good_data=True)